In [13]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch3. 연관분석</font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이의 자주 발생하는 속성을 찾고, 그 속성들 사이의 연관성이 어느 정도 있는지를 분석
- 활용분야 : 상품진열, 사기보험적발, 신상품 카테고리 구성,...
```
조건(left-hand side, item_base) : 오렌지주스(x) => 결과(right-hand side, item_add) : 와인(y)

연관분석 지표
1. 지지도(support) : 전체 데이터 중, 조건과 결과 항목들이 포함된 거래 비율(함께 얼마나 자주 나타나는지)
    (x, y)의 항목수 / 전체 데이터 수 = 0.2
2. 신뢰도(confidence) : 조건(x)이 발생했을 때, 결과가 동시에 일어날 확률(조건이 오면 얼마나 자주 결과가 오는지)
    (x=>y)의 항목수 / x가 나오는 항목수 = 0.5
3. 향상도(fit) : 우연히 발생할 규칙은 아니었는지 확인
    1미만 : 독립적으로 나오는 것보다 함께 나타날 가능성이 낮다
    1    : 서로 독립적. 아무 연관성 없다
    1초과 : 양의 상관관계(같이 잘 나온다)
    (x=>y)의 지지도 / x의 지지도*y의 지지도 = 0.2 / (0.4*0.6) = 0.2/0.24 = 0.83333
    
```
# 2. 연관분석 구현

In [14]:
import csv
with open('data/cf_basket.csv', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [15]:
from apyori import apriori
rules = apriori(transaction,
              min_support=0.15,
              min_confidence=0.1,
              min_lift=1.001)
rules = list(rules)
len(rules)

6

In [16]:
rule = rules[5]
rule

RelationRecord(items=frozenset({'소주', '와인', '콜라'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주', '와인'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'소주', '와인'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25)])

In [17]:
support = rule[1]
ordered_st = rule[2]
for item in ordered_st:
    # print(item)
    lhs = item[0]
    lhs = ','.join([x for x in lhs])
    rhs = item[1]
    rhs = ','.join([x for x in rhs])
    confidence = item[2]
    lift = item[3]
    print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")

콜라=>소주,와인 	 0.2 	 0.25 	 1.25
소주,와인=>콜라 	 0.2 	 1.0 	 1.25


In [18]:
rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
        # print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#         rules_lst.append({'lhs':lhs,
#                          'rhs':rhs,
#                          'support':support,
#                          'confidence':round(confidence,2),
#                          'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence','lift'])

,lhs,rhs,support,confidence,lift
0,맥주,콜라,0.4,1.00,1.25
1,콜라,맥주,0.4,0.50,1.25
2,소주,콜라,0.6,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
4,콜라,"소주,맥주",0.2,0.25,1.25
5,"소주,맥주",콜라,0.2,1.00,1.25
6,맥주,"와인,콜라",0.2,0.50,1.25
7,콜라,"와인,맥주",0.2,0.25,1.25
8,"와인,맥주",콜라,0.2,1.00,1.25
9,"와인,콜라",맥주,0.2,0.50,1.25


# 3. 뉴스 연관분석
- 경제뉴스 20개를 각각 명사만 가져와서 list (naver API) -> 연관분석
  ```
      [['단어1', '단어2', '단어3', ...],
       ['단어4', '단어5', '단어6', ...],
       ['단어2', '단어7', '단어1', ...]...]
    ```

In [19]:
from dotenv import load_dotenv
import os
load_dotenv() # .env의 시스템환경변수 불러오기

True

In [20]:
import os
import requests
import json
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

url = f"https://openapi.naver.com/v1/search/news.json" # JSON 결과
params = {
    'query':'경제',
    'display':20, # 가져올 데이터 갯수(기본값은 10)
    'sort':'date' # 최신순 뉴스
        }
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items'] # response를 json형태로 변환한 것중 'items'

news_texts = []
# 제목(title) + ' ' + 요약(description) 텍스트만 추출하여 list
for item in items:
    title = item.get('title').replace('<b>', ' ').replace('</b>', ' ')
    description = item.get('description').replace('<b>', ' ').replace('</b', ' ')
    news_texts.append(title + ' ' + description)
news_texts[:3], len(news_texts)

(['케어원·사회복지법인 다하, ESG 상생협력 업무협약 케어원과 다하는 최근 협약식을 갖고 장애인의 지속가능한 일자리 창출과  경제 >적 자립, 사회참여 확대를 위해 협력하기로 했다. 이번 협약에 따라 세하앤은 케어원의 초파리 트랩에 사용되는 유인제 생산을 비롯해 제품... ',
  '버려지던 전복 껍데기, 산란계 사료로 재활용 박수진 원장은 “지역에서 버려지는 자원을 새로운 가치로 전환하고 우리원의 전문성과 대학·기업의 역량을 연결했다는 데 의미가 있다”며 “앞으로도 지역 경제 > 활성화에 실질적으로 도움이 되는 협력사업을 지속... ',
  '옥천군 통합돌봄 대상자 집안 정리도 해준다 통합돌봄 대상자 가운데  경제 >적인 여유로 기초연금 등을 받지 못하는 경우는 본인 부담금이 발생한다. 군은 수행기관인 옥천지역자활센터에 보조금을 지원한다. 군 관계자는 “읍·면 및 통합돌봄 관계기관과 협력해... '],
 20)

In [21]:
# 명사추출(news_texts)
stopwords = {'기사', '기자'}
from konlpy.tag import Hannanum, Kkma, Komoran, Okt
from mecab import MeCab
analyzer = MeCab()
news = []
for article in news_texts:
    noun_list = analyzer.nouns(article)
    noun_list = [word for word, tag in analyzer.pos(article)\
                 if tag in ('NNG','NNP') and # 명사
                    word not in stopwords and # 불용어(stopwords) 제외
                    len(word)>1 ] # 두 글자 이상의 단어
    news.append(noun_list)
print(news[:3])

[['어원', '사회', '복지', '법인', '상생', '협력', '업무', '협약', '어원', '최근', '약식', '장애인', '지속', '가능', '일자리', '창출', '경제', '자립', '사회', '참여', '확대', '협력', '이번', '협약', '세하', '케어', '초파리', '트랩', '사용', '유인제', '생산', '제품'], ['전복', '껍데기', '산란계', '사료', '활용', '박수진', '원장', '지역', '자원', '가치', '전환', '전문', '대학', '기업', '역량', '연결', '의미', '지역', '경제', '활성', '실질', '도움', '협력', '사업', '지속'], ['옥천군', '통합', '대상자', '집안', '정리', '통합', '대상자', '가운데', '경제', '여유', '기초', '연금', '경우', '본인', '부담금', '발생', '수행', '기관', '옥천', '지역', '자활', '센터', '보조금', '지원', '관계자', '통합', '관계', '기관', '협력']]


In [23]:
rules = apriori(news,
               min_support=0.15,
               min_confidence=0.1,
               min_lift=1.000001)
rules = list(rules)
len(rules)

10

In [25]:
rules_lst = [] # 규칙을 저장할 dict list
for rule in rules:
    support = rule[1]
    ordered_st = rule[2]
    for item in ordered_st:
        # print(item)
        lhs = item[0]
        lhs = ','.join([x for x in lhs])
        rhs = item[1]
        rhs = ','.join([x for x in rhs])
        confidence = item[2]
        lift = item[3]
        # print(f"{lhs}=>{rhs} \t {support} \t {round(confidence,2)} \t {round(lift,2)}")
#         rules_lst.append({'lhs':lhs,
#                          'rhs':rhs,
#                          'support':support,
#                          'confidence':round(confidence,2),
#                          'lift':round(lift,2)})
        rules_lst.append([lhs, rhs, support, round(confidence, 2), round(lift, 2)])
import pandas as pd
df = pd.DataFrame(rules_lst, columns=['lhs','rhs','support','confidence','lift'])

In [29]:
df.sort_values(by=['lift', 'confidence', 'support'], ascending=False, inplace=True)
df # 지지도는 신뢰성 체크용

,lhs,rhs,support,confidence,lift
14,시장,"지역,경제",0.15,1.00,2.22
27,활성,"지역,경제",0.15,1.00,2.22
20,"경제,유치",지역,0.15,1.00,2.22
18,유치,"지역,경제",0.15,1.00,2.22
16,"경제,시장",지역,0.15,1.00,2.22
29,"경제,활성",지역,0.15,1.00,2.22
2,시장,지역,0.15,1.00,2.22
4,유치,지역,0.15,1.00,2.22
9,활성,지역,0.15,1.00,2.22
15,지역,"경제,시장",0.15,0.33,2.22
